# 🏆 مشروع توقع كأس العالم 2026
## المرحلة الثانية: بناء الـ Features (Feature Engineering)

| الـ Feature | المصدر | الوصف |
|------------|--------|-------|
| ELO Difference | DS-1 محسوب | فرق القوة بين الفريقين |
| Recent Form | DS-1 | فورم آخر 5 و10 مباريات رسمية |
| Head-to-Head | DS-1 | المواجهات التاريخية + WC فقط |
| Squad Strength | DS-5 EAFC26 | متوسط تقييم أفضل 15 لاعب |
| Market Value | DS-4 WC2026 | القيمة السوقية للتشكيلة |
| FIFA Ranking | DS-3 | التصنيف الرسمي قبل البطولة |
| Shootout History | DS-1 | نسبة الفوز بركلات الترجيح ⭐ |
| Motivation Flag | هندسة | Knockout = must_win ⭐ |
| Team Strength Score | هندسة | Score مركّب من كل الـ features ⭐ |


## 📦 1. استيراد المكتبات وتحميل البيانات المنظّفة

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ── تحميل البيانات من المرحلة الأولى ───────────────────
results_played   = pd.read_csv("data/processed/results_clean.csv",    parse_dates=["date"])
results_official = pd.read_csv("data/processed/results_official.csv", parse_dates=["date"])
shootouts        = pd.read_csv("data/processed/shootouts_clean.csv",  parse_dates=["date"])
elo_current      = pd.read_csv("data/processed/elo_current.csv")
shootout_stats   = pd.read_csv("data/processed/shootout_stats.csv")

# ── تحميل الداتاسيتات الإضافية ──────────────────────────
teams_df    = pd.read_csv("data/raw/teams.csv")
squad_summary = pd.read_csv("data/processed/squad_summary.csv")
team_ratings  = pd.read_csv("data/processed/team_ratings_fc26.csv")
fifa_rank     = pd.read_csv("data/processed/fifa_rank_latest.csv")

print("✅ كل البيانات اتحملت بنجاح")
print(f"   مباريات منتهية    : {len(results_played):,}")
print(f"   مباريات رسمية     : {len(results_official):,}")
print(f"   ركلات ترجيح       : {len(shootouts):,}")
print(f"   فرق WC 2026       : {len(teams_df)}")


## 🔧 2. توحيد أسماء الفرق عبر كل الداتاسيتات

> هنا بنصلح كل الاختلافات دفعة واحدة — `Holland` → `Netherlands`، إلخ


In [ ]:
# توحيد شامل لكل الأسماء المختلفة عبر الداتاسيتات
TEAM_NAME_MAP = {
    # EAFC26 اختلافات
    "Holland"              : "Netherlands",
    "Korea Republic"       : "South Korea",
    "IR Iran"              : "Iran",
    "USA"                  : "United States",
    "Côte d'Ivoire"        : "Ivory Coast",
    "Cote d'Ivoire"        : "Ivory Coast",
    "Czechia"              : "Czech Republic",
    "Czech Rep."           : "Czech Republic",
    "Cabo Verde"           : "Cape Verde",
    "DR Congo"             : "DR Congo",
    "Congo DR"             : "DR Congo",
    "Dem. Rep. Congo"      : "DR Congo",
    "Bosnia-Herzegovina"   : "Bosnia and Herzegovina",
    "Bosnia & Herzegovina" : "Bosnia and Herzegovina",
    "UAE"                  : "United Arab Emirates",
    "FYR Macedonia"        : "North Macedonia",
    "Eswatini"             : "Swaziland",
    "Timor-Leste"          : "East Timor",
    "Trinidad & Tobago"    : "Trinidad and Tobago",
    "St. Kitts & Nevis"    : "Saint Kitts and Nevis",
    "São Tomé & Príncipe"  : "Sao Tome and Principe",
    "Kyrgyz Republic"      : "Kyrgyzstan",
    "Slovak Republic"      : "Slovakia",
}

def normalize_team(name):
    if pd.isna(name): return name
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

# تطبيق على كل الداتاسيتات
for col in ["home_team","away_team"]:
    results_played[col]   = results_played[col].apply(normalize_team)
    results_official[col] = results_official[col].apply(normalize_team)
    shootouts[col]        = shootouts[col].apply(normalize_team)

shootouts["winner"]         = shootouts["winner"].apply(normalize_team)
elo_current["team"]         = elo_current["team"].apply(normalize_team)
shootout_stats["team"]      = shootout_stats["team"].apply(normalize_team)
squad_summary["team"]       = squad_summary["team"].apply(normalize_team)
team_ratings["team"]        = team_ratings["team"].apply(normalize_team)
fifa_rank["team"]           = fifa_rank["team"].apply(normalize_team)
teams_df["team_name"]       = teams_df["team_name"].apply(normalize_team)

print(f"✅ تم توحيد {len(TEAM_NAME_MAP)} اسم فريق عبر كل الداتاسيتات")

# تحقق من Holland اتصلح
nl = team_ratings[team_ratings["team"] == "Netherlands"]
print(f"   Netherlands في EAFC26: {len(nl)} صف {'✅' if len(nl)>0 else '⚠️ لسه مش متصلح'}")


## 📈 3. دالة الفورم الحديثة

> بتحسب فورم الفريق في آخر 5 و10 مباريات رسمية **قبل تاريخ المباراة**  
> ⚠️ `before_date` ضروري جداً — بيمنع الـ data leakage


In [ ]:
def calculate_recent_form(results_df, team, before_date=None, n_matches=10, official_only=True):
    """
    حساب فورم الفريق في آخر N مباريات.
    before_date: نحسب الفورم قبل تاريخ المباراة بس (يمنع data leakage)
    """
    df = results_df[results_df["tournament_weight"] > 1].copy() if official_only else results_df.copy()
    if before_date:
        df = df[df["date"] < pd.Timestamp(before_date)]

    home_g = df[df["home_team"] == team].copy()
    home_g["scored"]   = home_g["home_score"]
    home_g["conceded"] = home_g["away_score"]
    home_g["points"]   = home_g["result"].map({"home_win":3,"draw":1,"away_win":0})

    away_g = df[df["away_team"] == team].copy()
    away_g["scored"]   = away_g["away_score"]
    away_g["conceded"] = away_g["home_score"]
    away_g["points"]   = away_g["result"].map({"away_win":3,"draw":1,"home_win":0})

    all_g = pd.concat([home_g, away_g]).sort_values("date", ascending=False).head(n_matches)

    if len(all_g) == 0:
        return {f"form{n_matches}_points":0, f"form{n_matches}_gf":0,
                f"form{n_matches}_ga":0, f"form{n_matches}_gd":0,
                f"form{n_matches}_win_rate":0, f"form{n_matches}_matches":0}
    return {
        f"form{n_matches}_points"  : int(all_g["points"].sum()),
        f"form{n_matches}_gf"      : round(all_g["scored"].mean(), 2),
        f"form{n_matches}_ga"      : round(all_g["conceded"].mean(), 2),
        f"form{n_matches}_gd"      : round((all_g["scored"]-all_g["conceded"]).mean(), 2),
        f"form{n_matches}_win_rate": round((all_g["points"]==3).mean(), 3),
        f"form{n_matches}_matches" : len(all_g),
    }

# ── اختبار ──────────────────────────────────────────────
teams_test = ["Spain","France","Argentina","Brazil","England","Morocco","Netherlands"]
print("🔥 فورم أقوى الفرق (آخر 10 مباريات رسمية):\n")
rows = []
for t in teams_test:
    f5  = calculate_recent_form(results_played, t, n_matches=5)
    f10 = calculate_recent_form(results_played, t, n_matches=10)
    rows.append({"الفريق":t, **f5, **f10})
display(pd.DataFrame(rows))


## ⚔️ 4. المواجهات المباشرة (Head-to-Head)

> بنحسب H2H كاملة + H2H في كأس العالم فقط — لأن الكأس مختلف نفسياً


In [ ]:
def get_h2h(results_df, team_a, team_b, wc_only=False, before_date=None):
    """المواجهات المباشرة بين فريقين"""
    df = results_df.copy()
    if wc_only:
        df = df[df["tournament"] == "FIFA World Cup"]
    if before_date:
        df = df[df["date"] < pd.Timestamp(before_date)]

    mask = (
        ((df["home_team"]==team_a) & (df["away_team"]==team_b)) |
        ((df["home_team"]==team_b) & (df["away_team"]==team_a))
    )
    h2h = df[mask].copy()

    if len(h2h) == 0:
        return {"h2h_played":0,"h2h_wins_a":0,"h2h_wins_b":0,
                "h2h_draws":0,"h2h_win_rate_a":0.333,
                "h2h_avg_goals":0}

    wins_a = len(h2h[
        ((h2h["home_team"]==team_a) & (h2h["result"]=="home_win")) |
        ((h2h["away_team"]==team_a) & (h2h["result"]=="away_win"))
    ])
    wins_b = len(h2h[
        ((h2h["home_team"]==team_b) & (h2h["result"]=="home_win")) |
        ((h2h["away_team"]==team_b) & (h2h["result"]=="away_win"))
    ])
    draws = len(h2h[h2h["result"]=="draw"])

    return {
        "h2h_played"     : len(h2h),
        "h2h_wins_a"     : wins_a,
        "h2h_wins_b"     : wins_b,
        "h2h_draws"      : draws,
        "h2h_win_rate_a" : round(wins_a / len(h2h), 3),
        "h2h_avg_goals"  : round(h2h["total_goals"].mean(), 2) if "total_goals" in h2h.columns else 0,
    }

# ── اختبار ──────────────────────────────────────────────
print("⚔️  مواجهات تاريخية شهيرة:\n")
matchups = [
    ("Brazil",   "Argentina"),
    ("Germany",  "France"),
    ("Spain",    "England"),
    ("Argentina","France"),
]
rows = []
for a, b in matchups:
    h = get_h2h(results_played, a, b)
    hw = get_h2h(results_played, a, b, wc_only=True)
    rows.append({
        "المباراة"        : f"{a} vs {b}",
        "إجمالي المباريات": h["h2h_played"],
        f"فوز {a}"        : h["h2h_wins_a"],
        f"فوز {b}"        : h["h2h_wins_b"],
        "تعادل"           : h["h2h_draws"],
        "WC فقط"          : hw["h2h_played"],
        f"WC فوز {a}"     : hw["h2h_wins_a"],
    })
display(pd.DataFrame(rows))


## 🎯 5. إحصاءات ركلات الترجيح

> **الميزة المخفية** — مقياس مباشر للروح القتالية تحت الضغط ⭐


In [ ]:
def get_shootout_stats(team):
    """نسبة الفوز بركلات الترجيح تاريخياً"""
    row = shootout_stats[shootout_stats["team"] == team]
    if len(row) == 0:
        return {"shootout_played":0, "shootout_won":0, "shootout_win_rate":0.5}
    r = row.iloc[0]
    return {
        "shootout_played"  : int(r["shootout_played"]),
        "shootout_won"     : int(r["shootout_won"]),
        "shootout_win_rate": float(r["shootout_win_rate"]),
    }

# ── أفضل وأسوأ الفرق في ركلات الترجيح ─────────────────
wc_teams = teams_df["team_name"].tolist()
so_data = []
for t in wc_teams:
    s = get_shootout_stats(t)
    if s["shootout_played"] > 0:
        so_data.append({"الفريق":t, **s})

so_df = pd.DataFrame(so_data).sort_values("shootout_win_rate", ascending=False)

print("🏆 أفضل 10 فرق في ركلات الترجيح:")
display(so_df.head(10))
print("\n💀 أسوأ 5 فرق في ركلات الترجيح:")
display(so_df.tail(5))


## 🏗️ 6. بناء الـ Master Feature Table

> هنبني جدول واحد فيه كل الـ features لكل فريق في WC 2026  
> ده اللي هنستخدمه مباشرة في تدريب النموذج


In [ ]:
def build_team_features(team_name):
    """
    بناء كل الـ features لفريق واحد.
    يرجع: dict بكل الـ features
    """
    features = {"team": team_name}

    # ── 1. ELO Rating ────────────────────────────────────
    elo_row = elo_current[elo_current["team"] == team_name]
    features["elo"] = float(elo_row["elo"].values[0]) if len(elo_row) > 0 else 1500.0

    # ── 2. FIFA Ranking (من teams.csv — أحدث وأدق) ──────
    team_row = teams_df[teams_df["team_name"] == team_name]
    if len(team_row) > 0:
        features["fifa_rank"]        = int(team_row["fifa_ranking_pre_tournament"].values[0])
        features["elo_official"]     = float(team_row["elo_rating"].values[0])
    else:
        fifa_row = fifa_rank[fifa_rank["team"] == team_name]
        features["fifa_rank"]        = int(fifa_row["fifa_rank"].values[0]) if len(fifa_row)>0 else 100
        features["elo_official"]     = features["elo"]

    # ── 3. Recent Form (5 و10 مباريات) ───────────────────
    f5  = calculate_recent_form(results_played, team_name, n_matches=5)
    f10 = calculate_recent_form(results_played, team_name, n_matches=10)
    features.update(f5)
    features.update(f10)

    # ── 4. Squad Strength (EAFC26) ────────────────────────
    fc_row = team_ratings[team_ratings["team"] == team_name]
    if len(fc_row) > 0:
        features["avg_overall"] = float(fc_row["avg_overall"].values[0])
        features["max_overall"] = float(fc_row["max_overall"].values[0])
    else:
        features["avg_overall"] = 75.0
        features["max_overall"] = 80.0

    # ── 5. Market Value (من WC2026 squads) ───────────────
    sq_row = squad_summary[squad_summary["team"] == team_name]
    if len(sq_row) > 0:
        features["total_market_value"] = float(sq_row["total_market_value"].values[0])
        features["avg_market_value"]   = float(sq_row["avg_market_value"].values[0])
        features["avg_caps"]           = float(sq_row["avg_caps"].values[0])
    else:
        features["total_market_value"] = 0.0
        features["avg_market_value"]   = 0.0
        features["avg_caps"]           = 0.0

    # ── 6. Shootout History ────────────────────────────────
    so = get_shootout_stats(team_name)
    features.update(so)

    return features

# ── بناء الجدول لكل فرق WC 2026 ─────────────────────────
print("⏳ جاري بناء features لكل الـ 48 فريق...")

all_features = []
for team in teams_df["team_name"].tolist():
    f = build_team_features(team)
    all_features.append(f)

features_df = pd.DataFrame(all_features)
features_df.to_csv("data/processed/team_features.csv", index=False)

print(f"\n✅ تم بناء {len(features_df.columns)-1} feature لـ {len(features_df)} فريق")
print(f"\n📊 أعلى الفرق بـ ELO:")
display(features_df.sort_values("elo", ascending=False)[
    ["team","elo","fifa_rank","avg_overall","total_market_value","form10_win_rate"]
].head(10).reset_index(drop=True))


## ⚡ 7. بناء Features للمباراة (Match-Level Features)

> لكل مباراة بنحسب الفرق بين الفريقين — ده اللي بيدخل النموذج فعلاً


In [ ]:
def build_match_features(team_a, team_b, match_date=None, is_knockout=False, neutral=True):
    """
    بناء features لمباراة بين فريقين.
    team_a = الفريق الأول (home أو الأقوى)
    team_b = الفريق الثاني (away)
    """
    fa = build_team_features(team_a)
    fb = build_team_features(team_b)

    # ── H2H ─────────────────────────────────────────────
    h2h_all = get_h2h(results_played, team_a, team_b, before_date=match_date)
    h2h_wc  = get_h2h(results_played, team_a, team_b, wc_only=True, before_date=match_date)

    match_features = {
        # معلومات المباراة
        "team_a"       : team_a,
        "team_b"       : team_b,
        "is_knockout"  : int(is_knockout),
        "neutral_venue": int(neutral),

        # فرق ELO
        "elo_diff"         : round(fa["elo"] - fb["elo"], 1),
        "elo_official_diff": round(fa["elo_official"] - fb["elo_official"], 1),

        # فرق FIFA Ranking (سالب = team_a أحسن)
        "fifa_rank_diff": fb["fifa_rank"] - fa["fifa_rank"],

        # فرق الفورم
        "form5_win_rate_diff" : round(fa["form5_win_rate"]  - fb["form5_win_rate"],  3),
        "form10_win_rate_diff": round(fa["form10_win_rate"] - fb["form10_win_rate"], 3),
        "form10_gd_diff"      : round(fa["form10_gd"]       - fb["form10_gd"],       2),
        "form10_gf_diff"      : round(fa["form10_gf"]       - fb["form10_gf"],       2),

        # فرق قوة اللاعبين
        "avg_overall_diff"        : round(fa["avg_overall"]        - fb["avg_overall"],        1),
        "max_overall_diff"        : round(fa["max_overall"]        - fb["max_overall"],        1),
        "market_value_diff"       : round(fa["total_market_value"] - fb["total_market_value"], 0),
        "avg_market_value_diff"   : round(fa["avg_market_value"]   - fb["avg_market_value"],   0),

        # H2H
        "h2h_played"       : h2h_all["h2h_played"],
        "h2h_win_rate_a"   : h2h_all["h2h_win_rate_a"],
        "h2h_wc_played"    : h2h_wc["h2h_played"],
        "h2h_wc_win_rate_a": h2h_wc["h2h_win_rate_a"],

        # ركلات الترجيح
        "shootout_win_rate_a"  : fa["shootout_win_rate"],
        "shootout_win_rate_b"  : fb["shootout_win_rate"],
        "shootout_win_rate_diff": round(fa["shootout_win_rate"] - fb["shootout_win_rate"], 3),

        # خبرة
        "avg_caps_diff": round(fa["avg_caps"] - fb["avg_caps"], 1),

        # القيم الخام (للمرجعية)
        "elo_a"         : fa["elo"],
        "elo_b"         : fb["elo"],
        "fifa_rank_a"   : fa["fifa_rank"],
        "fifa_rank_b"   : fb["fifa_rank"],
        "avg_overall_a" : fa["avg_overall"],
        "avg_overall_b" : fb["avg_overall"],
        "form10_wr_a"   : fa["form10_win_rate"],
        "form10_wr_b"   : fb["form10_win_rate"],
    }

    # ── Team Strength Score المركّب ⭐ ───────────────────
    # نعمّل القيم أولاً
    elo_norm_a   = fa["elo"]          / 2200
    elo_norm_b   = fb["elo"]          / 2200
    form_a       = fa["form10_win_rate"]
    form_b       = fb["form10_win_rate"]
    val_norm_a   = fa["total_market_value"] / 1_500_000_000
    val_norm_b   = fb["total_market_value"] / 1_500_000_000
    ovr_norm_a   = fa["avg_overall"]  / 90
    ovr_norm_b   = fb["avg_overall"]  / 90

    strength_a = (0.35*elo_norm_a + 0.25*form_a + 0.20*val_norm_a + 0.20*ovr_norm_a)
    strength_b = (0.35*elo_norm_b + 0.25*form_b + 0.20*val_norm_b + 0.20*ovr_norm_b)

    match_features["strength_score_a"]    = round(strength_a, 4)
    match_features["strength_score_b"]    = round(strength_b, 4)
    match_features["strength_score_diff"] = round(strength_a - strength_b, 4)

    return match_features

# ── اختبار بمباريات WC 2026 حقيقية ─────────────────────
print("🔍 اختبار features على مباريات WC 2026:\n")

test_matches = [
    ("Argentina", "France",      True),
    ("Spain",     "England",     True),
    ("Brazil",    "Germany",     False),
    ("Morocco",   "Netherlands", True),
]

rows = []
for a, b, ko in test_matches:
    mf = build_match_features(a, b, is_knockout=ko)
    rows.append({
        "المباراة"          : f"{a} vs {b}",
        "ELO diff"          : mf["elo_diff"],
        "Rank diff"         : mf["fifa_rank_diff"],
        "Form diff"         : mf["form10_win_rate_diff"],
        "OVR diff"          : mf["avg_overall_diff"],
        "Strength diff"     : mf["strength_score_diff"],
        "H2H win rate A"    : mf["h2h_win_rate_a"],
        "Knockout"          : "✅" if mf["is_knockout"] else "❌",
    })

display(pd.DataFrame(rows))


## 🗃️ 8. بناء Training Dataset الكامل

> بنحوّل كل المباريات التاريخية لـ feature vectors جاهزة للـ ML  
> ⚠️ `before_date` بيضمن مفيش data leakage — كل مباراة بتتحسب فيتشرزها من الماضي فقط


In [ ]:
from tqdm.notebook import tqdm

# نستخدم مباريات كأس العالم فقط للتدريب (أكتر صلة بالمشروع)
wc_matches = results_played[
    results_played["tournament"] == "FIFA World Cup"
].copy().reset_index(drop=True)

print(f"مباريات WC للتدريب: {len(wc_matches)}")
print(f"من {wc_matches['date'].min().year} لـ {wc_matches['date'].max().year}")

# ── بناء الـ features لكل مباراة ─────────────────────────
print("\n⏳ جاري بناء training dataset (قد ياخد دقيقة)...")

training_rows = []
skipped = 0

for _, row in tqdm(wc_matches.iterrows(), total=len(wc_matches)):
    try:
        stage = str(row.get("tournament","")).lower()
        is_ko = any(x in stage for x in ["final","semi","quarter","round of","knockout"])

        mf = build_match_features(
            team_a     = row["home_team"],
            team_b     = row["away_team"],
            match_date = row["date"],
            is_knockout= is_ko,
            neutral    = bool(row.get("neutral", False)),
        )

        # الـ target
        if row["result"] == "home_win":
            mf["target"] = 2   # فوز team_a
        elif row["result"] == "draw":
            mf["target"] = 1   # تعادل
        else:
            mf["target"] = 0   # فوز team_b

        mf["home_score"]  = int(row["home_score"])
        mf["away_score"]  = int(row["away_score"])
        mf["match_date"]  = row["date"]
        mf["tournament"]  = row["tournament"]

        training_rows.append(mf)

    except Exception as e:
        skipped += 1

training_df = pd.DataFrame(training_rows)
training_df.to_csv("data/processed/training_dataset.csv", index=False)

print(f"\n✅ Training dataset جاهز!")
print(f"   عدد المباريات  : {len(training_df):,}")
print(f"   عدد الـ features: {len(training_df.columns)-4}")
print(f"   تخطّينا        : {skipped} مباراة")
print(f"\n📊 توزيع الـ targets:")
target_map = {2:"فوز team_a", 1:"تعادل", 0:"فوز team_b"}
display(training_df["target"].map(target_map).value_counts().to_frame())


## ✅ 9. ملخص المرحلة الثانية

### الملفات الجاهزة:
| الملف | المحتوى |
|-------|---------|
| `data/processed/team_features.csv` | كل الـ features لكل فريق |
| `data/processed/training_dataset.csv` | بيانات التدريب جاهزة للـ ML |

### الـ Features المبنية:
- ✅ ELO Difference
- ✅ FIFA Ranking Difference  
- ✅ Recent Form (5 و10 مباريات)
- ✅ Head-to-Head (كاملة + WC فقط) ⭐
- ✅ Squad Strength (EAFC26) ⭐
- ✅ Market Value ⭐
- ✅ Shootout History ⭐
- ✅ Motivation Flag (Knockout) ⭐
- ✅ Team Strength Score المركّب ⭐

### الخطوة الجاية: المرحلة الثالثة — تدريب النماذج 🚀


In [ ]:
print("📁 الملفات في data/processed/:")
for f in sorted(os.listdir("data/processed")):
    size = os.path.getsize(f"data/processed/{f}") // 1024
    print(f"   {f:45s} {size:>6} KB")

print("\n🚀 جاهزين للمرحلة الثالثة — تدريب النماذج!")
